In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, random_split
import json
import csv
import os


os.makedirs("artifacts/figures", exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Torch version:', torch.__version__)

Device: cuda
Torch version: 2.7.1+cu118


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.EMNIST(
    root="./data",
    split="balanced",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.EMNIST(
    root="./data",
    split="balanced",
    train=False,
    download=True,
    transform=transform
)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_dataset, [train_size, val_size], generator=generator)

BATCH_SIZE = 128

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

x, y = next(iter(train_loader))
print(x.shape, y.shape)
print(x.min(), x.max())

torch.Size([128, 1, 28, 28]) torch.Size([128])
tensor(0.) tensor(1.)


In [3]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout=0.0, batchnorm=False):
        super().__init__()
        
        layers = []
        input_size = 28 * 28
        
        for h in hidden_sizes:
            layers.append(nn.Linear(input_size, h))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_size = h
        
        layers.append(nn.Linear(input_size, 47))  # EMNIST balanced = 47 классов
        
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

In [4]:
def accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += accuracy(logits, y)
    
    return total_loss / len(loader), total_acc / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        logits = model(x)
        loss = criterion(logits, y)
        
        total_loss += loss.item()
        total_acc += accuracy(logits, y)
    
    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
def train_model(model, epochs, early_stopping=False, patience=3):
    model.to(device)
    
    optimizer = optim.Adam(model.parameters())
    criterion = nn.CrossEntropyLoss()
    
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    best_val_acc = 0
    best_epoch = 0
    patience_counter = 0
    
    print(f"Start training for {epochs} epochs | early_stopping={early_stopping}")
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_accuracy)
        
        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
            f"train_acc={train_acc:.4f} val_acc={val_accuracy:.4f}"
        )
        
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            best_epoch = epoch
            best_weights = model.state_dict()
            patience_counter = 0
            print(f"New best val_acc: {best_val_acc:.4f} (epoch {epoch+1})")
        else:
            patience_counter += 1
            print(f"patience: {patience_counter}/{patience}")
        
        if early_stopping and patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_weights)
    
    print(f"Training finished. Best val_acc={best_val_acc:.4f} at epoch {best_epoch+1}")
    
    return history, best_val_acc, best_epoch, epoch+1

In [6]:
runs = []

def run_experiment(exp_id, hidden, dropout, batchnorm, early_stop=False):
    print(f"\n===== START {exp_id} =====")
    print(f"Config: hidden={hidden}, dropout={dropout}, batchnorm={batchnorm}, early_stop={early_stop}")
    
    model = MLP(hidden, dropout=dropout, batchnorm=batchnorm)
    
    history, best_val_acc, best_epoch, epochs_trained = train_model(
        model, epochs=15, early_stopping=early_stop, patience=4
    )
    
    print(f"Finished {exp_id}:")
    print(f"  epochs_trained = {epochs_trained}")
    print(f"  best_val_acc   = {best_val_acc:.4f}")
    print(f"  best_epoch     = {best_epoch}")
    
    runs.append({
        "experiment_id": exp_id,
        "dataset": "EMNIST",
        "seed": SEED,
        "model_summary": str(hidden),
        "dropout": dropout,
        "batchnorm": batchnorm,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": min(history["val_loss"])
    })
    
    print(f"===== END {exp_id} =====\n")
    
    return model, history, best_val_acc


# E0
m0, h0, a0 = run_experiment("E0", [128], 0.0, False)

# E1
m1, h1, a1 = run_experiment("E1", [256,128], 0.0, False)

# E2
m2, h2, a2 = run_experiment("E2", [256,128], 0.3, False)

# E3
m3, h3, a3 = run_experiment("E3", [256,128], 0.0, True)


===== START E0 =====
Config: hidden=[128], dropout=0.0, batchnorm=False, early_stop=False
Start training for 15 epochs | early_stopping=False
Epoch 01/15 | train_loss=1.4510 val_loss=1.0849 | train_acc=0.6111 val_acc=0.6949
New best val_acc: 0.6949 (epoch 1)
Epoch 02/15 | train_loss=0.9367 val_loss=0.8603 | train_acc=0.7307 val_acc=0.7522
New best val_acc: 0.7522 (epoch 2)
Epoch 03/15 | train_loss=0.7662 val_loss=0.7483 | train_acc=0.7699 val_acc=0.7795
New best val_acc: 0.7795 (epoch 3)
Epoch 04/15 | train_loss=0.6709 val_loss=0.6937 | train_acc=0.7942 val_acc=0.7907
New best val_acc: 0.7907 (epoch 4)
Epoch 05/15 | train_loss=0.6125 val_loss=0.6448 | train_acc=0.8103 val_acc=0.8027
New best val_acc: 0.8027 (epoch 5)
Epoch 06/15 | train_loss=0.5680 val_loss=0.6156 | train_acc=0.8197 val_acc=0.8114
New best val_acc: 0.8114 (epoch 6)
Epoch 07/15 | train_loss=0.5336 val_loss=0.6005 | train_acc=0.8282 val_acc=0.8138
New best val_acc: 0.8138 (epoch 7)
Epoch 08/15 | train_loss=0.5064 val_lo

In [7]:
best_exp = max(runs, key=lambda x: x["best_val_accuracy"])
print(best_exp)

if best_exp["experiment_id"] == "E2":
    best_hidden, best_dropout, best_bn = [256,128], 0.3, False
else:
    best_hidden, best_dropout, best_bn = [256,128], 0.0, True

best_model = MLP(best_hidden, dropout=best_dropout, batchnorm=best_bn)

best_model, best_history, best_acc = None, None, None

best_model, best_history, best_acc = run_experiment(
    "E4", best_hidden, best_dropout, best_bn, early_stop=True
)

{'experiment_id': 'E3', 'dataset': 'EMNIST', 'seed': 42, 'model_summary': '[256, 128]', 'dropout': 0.0, 'batchnorm': True, 'epochs_trained': 15, 'best_val_accuracy': 0.8490466101694916, 'best_val_loss': 0.4615215329968997}

===== START E4 =====
Config: hidden=[256, 128], dropout=0.0, batchnorm=True, early_stop=True
Start training for 15 epochs | early_stopping=True
Epoch 01/15 | train_loss=0.9921 val_loss=0.5985 | train_acc=0.7289 val_acc=0.8093
New best val_acc: 0.8093 (epoch 1)
Epoch 02/15 | train_loss=0.5291 val_loss=0.5196 | train_acc=0.8242 val_acc=0.8306
New best val_acc: 0.8306 (epoch 2)
Epoch 03/15 | train_loss=0.4472 val_loss=0.4870 | train_acc=0.8466 val_acc=0.8363
New best val_acc: 0.8363 (epoch 3)
Epoch 04/15 | train_loss=0.4012 val_loss=0.4716 | train_acc=0.8595 val_acc=0.8423
New best val_acc: 0.8423 (epoch 4)
Epoch 05/15 | train_loss=0.3700 val_loss=0.4599 | train_acc=0.8686 val_acc=0.8456
New best val_acc: 0.8456 (epoch 5)
Epoch 06/15 | train_loss=0.3439 val_loss=0.4569

In [8]:
criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, test_loader, criterion)

print("TEST ACC:", test_acc)

TEST ACC: 0.8423681976843853


In [9]:
with open("artifacts/runs.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=runs[0].keys())
    writer.writeheader()
    writer.writerows(runs)

torch.save(best_model.state_dict(), "artifacts/best_model.pt")

config = {
    "hidden": best_hidden,
    "dropout": best_dropout,
    "batchnorm": best_bn,
    "seed": SEED,
    "dataset": "EMNIST"
}

with open("artifacts/best_config.json", "w") as f:
    json.dump(config, f, indent=4)

plt.plot(best_history["train_loss"], label="train_loss")
plt.plot(best_history["val_loss"], label="val_loss")
plt.legend()
plt.savefig("artifacts/figures/curves_best.png")
plt.close()

In [39]:
from PIL import Image

model = MLP(hidden_sizes=[256,128], dropout=0.0, batchnorm=True)

model.load_state_dict(torch.load("artifacts/best_model.pt", map_location=device))
model.to(device)
model.eval()


transform = transforms.Compose([
    transforms.Grayscale(), 
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])


def predict_image(path):
    img = Image.open(path)

    img = img.rotate(90, expand=True)
    img = img.transpose(Image.FLIP_LEFT_RIGHT)

    img = transform(img)
    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img)
        pred = torch.argmax(logits, dim=1).item()

    return pred


In [40]:
classes = train_dataset.classes

pred = predict_image("test.png")
symbol = classes[pred]

print("Predicted:", symbol)

Predicted: 3
